In [1]:
import os, sys, shutil
import numpy as np
import pandas as sp
import pandas as pd
import nibabel as nib
import json

In [2]:
home = os.path.expanduser("~")
print(home)
atlas_dir = os.path.join(home, 'research_projects/snaplab_tools/data/atlases')

/home/lindenmp


In [3]:
in_dir_cere = os.path.join(atlas_dir, 'MDTB10')

for n_regions in [100, 200, 400]:
    for tian_scale in [1, 2, 3, 4]:
        in_dir = os.path.join(atlas_dir, 'SchaeferMSA/atlas-Schaefer{0}7MSA{1}'.format(n_regions, tian_scale))
        out_dir = os.path.join(atlas_dir, 'SchaeferMSAMDTB10/atlas-Schaefer{0}7MSA{1}MDTB10'.format(n_regions, tian_scale))
        if os.path.isdir(out_dir):
            shutil.rmtree(out_dir)
        os.makedirs(out_dir)

        for space in ['MNI152NLin2009cAsym', 'MNI152NLin6Asym']:
            for res in [1, 2]:
                print(n_regions, tian_scale, space, res)
                parc_file = os.path.join(in_dir, 'atlas-Schaefer{0}7MSA{1}_space-{2}_res-0{3}_dseg.nii.gz'.format(n_regions, tian_scale, space, res))
                cere_file = os.path.join(in_dir_cere, 'atlas-MDTB10_space-{0}_res-0{1}_dseg.nii.gz'.format(space, res))

                # load parc file
                parc = nib.load(parc_file)
                parc_data = parc.get_fdata()
                cere = nib.load(cere_file)
                cere_data = cere.get_fdata()
                index_offset = parc_data.max()

                cere_data_mask = cere_data > 0  # create binary cerebellum mask
                parc_data[cere_data_mask] = 0  # prep parc data by zeroing out cerebellar voxels
                cere_data += index_offset  # offset cerebellum parc data
                cere_data[~cere_data_mask] = 0  # re-zero voxels in cerebellum
                parc_data += cere_data  # add cerebellum to parcellation

                # save out (overwrite)
                parc_out = nib.Nifti1Image(parc_data, affine=parc.affine, header=parc.header)
                parc_file_out = os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}MDTB10_space-{2}_res-0{3}_dseg.nii.gz'.format(n_regions, tian_scale, space, res))
                nib.save(parc_out, parc_file_out)

                json_file = os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}MDTB10_dseg.json'.format(n_regions, tian_scale))
                data = {"BIDSVersion": "1.8.0", "Name": "Schaefer{0}7MSA{1}MDTB10".format(n_regions, tian_scale)}  # , "DatasetType": "atlas"
                # creating a JSON string
                json_string = json.dumps(data)
                # storing it in a file
                with open(json_file, "w") as json_data:
                    json.dump(data, json_data)
         
        # update tsv file
        tsv_file = os.path.join(in_dir, 'atlas-Schaefer{0}7MSA{1}_dseg.tsv'.format(n_regions, tian_scale))
        df = pd.read_csv(tsv_file, header=0, index_col=0, sep='\t')
        tsv_file_cere = os.path.join(in_dir_cere, 'atlas-MDTB10_dseg.tsv')
        df_cere = pd.read_csv(tsv_file_cere, header=0, index_col=0, sep='\t')
        
        df['cerebellum'] = False
        df_cere.drop(columns=['color'], inplace=True)
        df_cere['cortex'] = False
        df_cere['subcortex'] = False
        df_cere['cerebellum'] = True
        df_cere['label'] = 'cerebellum_' + df_cere['label'].astype(str)
        df_cere.index = (df_cere.index.values + index_offset).astype(int)
        df_cere.index.name = 'index'

        df_out = pd.concat((df, df_cere), axis=0)
        tsv_file_out = os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}MDTB10_dseg.tsv'.format(n_regions, tian_scale))
        df_out.to_csv(tsv_file_out, sep="\t")

100 1 MNI152NLin2009cAsym 1
100 1 MNI152NLin2009cAsym 2
100 1 MNI152NLin6Asym 1
100 1 MNI152NLin6Asym 2
100 2 MNI152NLin2009cAsym 1
100 2 MNI152NLin2009cAsym 2
100 2 MNI152NLin6Asym 1
100 2 MNI152NLin6Asym 2
100 3 MNI152NLin2009cAsym 1
100 3 MNI152NLin2009cAsym 2
100 3 MNI152NLin6Asym 1
100 3 MNI152NLin6Asym 2
100 4 MNI152NLin2009cAsym 1
100 4 MNI152NLin2009cAsym 2
100 4 MNI152NLin6Asym 1
100 4 MNI152NLin6Asym 2
200 1 MNI152NLin2009cAsym 1
200 1 MNI152NLin2009cAsym 2
200 1 MNI152NLin6Asym 1
200 1 MNI152NLin6Asym 2
200 2 MNI152NLin2009cAsym 1
200 2 MNI152NLin2009cAsym 2
200 2 MNI152NLin6Asym 1
200 2 MNI152NLin6Asym 2
200 3 MNI152NLin2009cAsym 1
200 3 MNI152NLin2009cAsym 2
200 3 MNI152NLin6Asym 1
200 3 MNI152NLin6Asym 2
200 4 MNI152NLin2009cAsym 1
200 4 MNI152NLin2009cAsym 2
200 4 MNI152NLin6Asym 1
200 4 MNI152NLin6Asym 2
400 1 MNI152NLin2009cAsym 1
400 1 MNI152NLin2009cAsym 2
400 1 MNI152NLin6Asym 1
400 1 MNI152NLin6Asym 2
400 2 MNI152NLin2009cAsym 1
400 2 MNI152NLin2009cAsym 2
400 2 MN

## Copy dataset_description

In [4]:
in_file = os.path.join(atlas_dir, 'dataset_description.json')
out_file = os.path.join(atlas_dir, 'SchaeferMSAMDTB10', 'dataset_description.json')
shutil.copyfile(in_file, out_file)

'/home/lindenmp/research_projects/snaplab_tools/data/atlases/SchaeferMSAMDTB10/dataset_description.json'